In [ ]:
# replicate_brent_comparison.ipynb
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, classification_report, precision_recall_curve
from sklearn.preprocessing import StandardScaler

# 1. LOAD DATA
DATA_PATH = 'C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/final_dataset.parquet'

def load_replication_data():
    df = pd.read_parquet(DATA_PATH)
    target = 'Disrupted'
    return df, target

def run_experiment(df, target_col, feature_list, experiment_name):
    print(f"\n{'='*10} RUNNING: {experiment_name} {'='*10}")
    
    # Filter and prepare data
    features = [f for f in feature_list if f in df.columns]
    X = df[features].fillna(0)
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Class Weighting
    num_neg = (y_train == 0).sum()
    num_pos = (y_train == 1).sum()
    scale_weight = num_neg / num_pos 

    params = {
        'n_estimators': 200,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': scale_weight,
        'use_label_encoder': False,
        'eval_metric': 'logloss',
        'random_state': 42
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train_scaled, y_train)

    # Threshold Optimization
    y_probs = model.predict_proba(X_test_scaled)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    y_pred_optimized = (y_probs >= best_threshold).astype(int)

    # Collect Metrics
    metrics = {
        'Experiment': experiment_name,
        'F1-Score': best_f1,
        'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_optimized),
        'ROC-AUC': roc_auc_score(y_test, y_probs),
        'Best Threshold': best_threshold
    }
    
    print(f"Optimal Threshold: {best_threshold:.4f} | F1: {best_f1:.4f}")
    return metrics

if __name__ == "__main__":
    df, target = load_replication_data()
    
    # Define Feature Sets
    core_features = ['rides_planned', 'disruption_count', 'stop_count', 'DR', 'RH', 'SQ', 'TG', 'TN', 'TX', 'RHX']
    ses_features = [
        'target_SES_Score_Wealth_Avg', 'target_SES_Score_Education_Avg', 'target_TotalVandalism', 'target_Remoteness_Index',
        'source_SES_Score_Wealth_Avg', 'source_SES_Score_Education_Avg', 'source_TotalVandalism', 'source_Remoteness_Index'
    ]
    
    # Run both experiments
    baseline_results = run_experiment(df, target, core_features, "Standard Baseline")
    enhanced_results = run_experiment(df, target, core_features + ses_features, "Socioeconomic Enhanced")
    
    # Compare results side-by-side
    comparison_df = pd.DataFrame([baseline_results, enhanced_results]).set_index('Experiment')
    
    print("\n" + "#"*30)
    print("FINAL COMPARISON TABLE")
    print("#"*30)
    print(comparison_df.round(4))
    
    # Calculate the 'Lift'
    lift = ((comparison_df.loc['Socioeconomic Enhanced'] / comparison_df.loc['Standard Baseline']) - 1) * 100
    print("\nPercentage Lift from SES variables:")
    print(lift[['F1-Score', 'Balanced Accuracy', 'ROC-AUC']].round(2).astype(str) + '%')


========== RUNNING: Standard Baseline ==========


c:\MiniForge3\lib\site-packages\xgboost\training.py:199: UserWarning: [17:30:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Optimal Threshold: 0.5444 | F1: 0.0338

========== RUNNING: Socioeconomic Enhanced ==========


c:\MiniForge3\lib\site-packages\xgboost\training.py:199: UserWarning: [17:30:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Optimal Threshold: 0.8025 | F1: 0.2425

##############################
FINAL COMPARISON TABLE
##############################
                        F1-Score  Balanced Accuracy  ROC-AUC  Best Threshold
Experiment                                                                  
Standard Baseline         0.0338             0.5218   0.5562          0.5444
Socioeconomic Enhanced    0.2425             0.7142   0.8958          0.8025

Percentage Lift from SES variables:
F1-Score             618.26%
Balanced Accuracy     36.89%
ROC-AUC               61.05%
dtype: object


In [14]:
# replicate_brent_seasonal_final.ipynb
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler

DATA_PATH = 'C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/final_dataset.parquet'

def run_seasonal_experiment(df, target_col, core_features, ses_features, season=None):
    # Filter by season
    if season:
        print(f"\n--- MODELING SEASON: {season.upper()} ---")
        df_run = df[df['season'] == season].copy()
    else:
        print(f"\n--- MODELING GLOBAL DATA ---")
        df_run = df.copy()

    if len(df_run) == 0:
        print(f"Error: No data found for season: {season}")
        return {'F1': 0, 'AUC': 0, 'BA': 0}

    # Combine all available features
    all_features = core_features + ses_features
    available_features = [f for f in all_features if f in df_run.columns]
    
    X = df_run[available_features].fillna(0)
    y = df_run[target_col].astype(int)

    # Stratified split to maintain class balance
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Class weighting to handle imbalance
    scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.03,
        scale_pos_weight=scale_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
    model.fit(X_train_scaled, y_train)

    # Probabilities for threshold optimization
    y_probs = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate Precision-Recall curve to find the best F1 threshold
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)
    
    # Get the maximum F1 and corresponding threshold
    best_f1 = np.max(f1_scores)
    best_threshold = thresholds[np.argmax(f1_scores)]
    
    # Final predictions based on optimized threshold
    y_pred_final = (y_probs >= best_threshold).astype(int)
    
    return {
        'F1': best_f1,
        'AUC': roc_auc_score(y_test, y_probs),
        'BA': balanced_accuracy_score(y_test, y_pred_final)
    }

if __name__ == "__main__":
    # Load the dataset
    full_df = pd.read_parquet(DATA_PATH)

    # Define seasons based on date ranges
    def get_season(date):
        if pd.isna(date): return 'unknown'
        month, day = date.month, date.day
        if (month == 12 and day >= 21) or (month <= 3 and (month < 3 or day <= 20)):
            return 'winter'
        elif (month == 3 and day >= 21) or (month <= 6 and (month < 6 or day <= 20)):
            return 'spring'
        elif (month == 6 and day >= 21) or (month <= 9 and (month < 9 or day <= 20)):
            return 'summer'
        else:
            return 'autumn'

    # Ensure Date column is created correctly
    # If your index is the Date, use this:
    full_df['Date_Col'] = pd.to_datetime(full_df.index, errors='coerce')
    
    # Map the season
    full_df['season'] = full_df['Date_Col'].apply(get_season)
    
    # Verify season counts
    print("Samples per season:")
    print(full_df['season'].value_counts())

    core = ['rides_planned', 'disruption_count', 'stop_count', 'DR', 'RH', 'SQ', 'TG', 'TN', 'TX', 'RHX']
    ses = [
        'target_SES_Score_Wealth_Avg', 'target_SES_Score_Education_Avg', 'target_TotalVandalism', 'target_Remoteness_Index',
        'source_SES_Score_Wealth_Avg', 'source_SES_Score_Education_Avg', 'source_TotalVandalism', 'source_Remoteness_Index'
    ]

    # Run comparisons
    global_res = run_seasonal_experiment(full_df, 'Disrupted', core, ses, season=None)
    summer_res = run_seasonal_experiment(full_df, 'Disrupted', core, ses, season='summer')

    print("\n" + "="*30)
    print("FINAL REPLICATION RESULTS")
    print("="*30)
    print(f"Global Model F1: {global_res['F1']:.4f}")
    print(f"Summer Model F1: {summer_res['F1']:.4f} (Target: ~0.3)")
    print(f"Summer AUC:      {summer_res['AUC']:.4f}")
    print(f"Summer Bal Acc:  {summer_res['BA']:.4f}")

Samples per season:
season
winter    654436
Name: count, dtype: int64

--- MODELING GLOBAL DATA ---

--- MODELING SEASON: SUMMER ---
Error: No data found for season: summer

FINAL REPLICATION RESULTS
Global Model F1: 0.2408
Summer Model F1: 0.0000 (Target: ~0.3)
Summer AUC:      0.0000
Summer Bal Acc:  0.0000
